In [1]:
import gradio as gr
import numpy as np
import cv2
import tensorflow as tf
from PIL import Image


model = tf.keras.models.load_model('/kaggle/input/models/haricharanpampati/task1/keras/default/1/model.keras') #[cite: 2]

gender_dict = {0: 'Male', 1: 'Female'}

def detect_hair_length(image_pil):
    """
    OpenCV heuristic to estimate hair length.
    Detects the face, isolates the region below the face (shoulders), 
    and measures the edge density to classify 'Long' vs 'Short' hair.
    """

    img_rgb = np.array(image_pil.convert('RGB'))
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    
    if len(faces) == 0:
        return "Short"
        
    x, y, w, h = faces[0]
    
    img_height, img_width = gray.shape
    roi_top = min(y + h, img_height)
    roi_bottom = min(y + h + int(h * 0.8), img_height)
    roi_left = max(x - int(w * 0.2), 0)
    roi_right = min(x + w + int(w * 0.2), img_width)
    
    if roi_top >= roi_bottom:
        return "Short"
        
    shoulder_roi = gray[roi_top:roi_bottom, roi_left:roi_right]
    
    edges = cv2.Canny(shoulder_roi, 50, 150)
    edge_density = np.sum(edges / 255.0) / (shoulder_roi.shape[0] * shoulder_roi.shape[1] + 1e-5)
    
    if edge_density > 0.04:
        return "Long"
    else:
        return "Short"

def task1_pipeline(input_image):
    try:
        img_gray = input_image.convert('L')
        img_resized = img_gray.resize((128, 128), Image.Resampling.BILINEAR if hasattr(Image, 'Resampling') else Image.BILINEAR)
        img_array = np.array(img_resized).reshape(1, 128, 128, 1) / 255.0
        
        predictions = model.predict(img_array)
        
        pred_gender_prob = float(predictions[0][0][0])
        pred_age_norm = float(predictions[1][0][0])
        
        predicted_age = int(pred_age_norm * 100)
        base_gender = gender_dict[int(np.round(pred_gender_prob))]
        
        hair_length = detect_hair_length(input_image)
        
        final_gender = base_gender
        
        if 20 <= predicted_age <= 30:
            if hair_length == "Long":
                final_gender = "Female" 
            else:
                final_gender = "Male" 
                

        return str(predicted_age), base_gender, hair_length, final_gender
        
    except Exception as e:
        error_msg = f"Exception: {str(e)}"
        return error_msg, error_msg, error_msg, error_msg

gui = gr.Interface(
    fn=task1_pipeline,
    inputs=gr.Image(type="pil", label="Upload Person Image"),
    outputs=[
        gr.Textbox(label="Predicted Age"),
        gr.Textbox(label="Base Model Gender"),
        gr.Textbox(label="Detected Hair Length"),
        gr.Textbox(label="Final Task 1 Gender (Adjusted)")
    ],
    title="Task 1: Long Hair Identification",
    description="Upload an image to predict age and gender. If the user is aged 20-30, gender prediction is overridden based on hair length." #[cite: 5]
)

gui.launch(share=True)

2026-10-25 18:32:27.881837: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c09dedf2956acf7330.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [2]:
#task 2
import os
from datetime import datetime
import cv2
import gradio as gr
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf

MODEL_PATH = "/kaggle/input/models/haricharanpampati/task1/keras/default/1/model.keras"
GENDER_MAP = {0: "Male", 1: "Female"}

if os.path.exists(MODEL_PATH):
  base_model = tf.keras.models.load_model(MODEL_PATH)
  print(f"[INFO] Loaded trained base model: {MODEL_PATH}")
else:
  inputs = tf.keras.Input(shape=(128, 128, 1))
  x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu")(inputs)
  x = tf.keras.layers.GlobalAveragePooling2D()(x)
  g_out = tf.keras.layers.Dense(1, activation="sigmoid", name="gender_out")(x)
  a_out = tf.keras.layers.Dense(1, activation="linear", name="age_out")(x)
  base_model = tf.keras.Model(inputs=inputs, outputs=[g_out, a_out])
  print("[WARN] Model file not found. Initialized scaffold model.")

LOG_CSV_PATH = "visitor_logs.csv"
LOG_XLSX_PATH = "visitor_logs.xlsx"

if not os.path.exists(LOG_CSV_PATH):
  df_init = pd.DataFrame(
      columns=["Timestamp", "Visitor_ID", "Age", "Gender", "Category"]
  )
  df_init.to_csv(LOG_CSV_PATH, index=False)
  df_init.to_excel(LOG_XLSX_PATH, index=False)


def preprocess_face(face_crop):
  """Preprocess a cropped face to 128x128 grayscale normalized array."""
  img = Image.fromarray(face_crop).convert("L")
  img = img.resize((128, 128), Image.Resampling.BILINEAR)
  arr = np.array(img, dtype=np.float32).reshape(1, 128, 128, 1) / 255.0
  return arr


def log_visitor(visitor_id, age, gender, category):
  """Logs unique visitor record with timestamp."""
  timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
  new_entry = {
      "Timestamp": timestamp,
      "Visitor_ID": visitor_id,
      "Age": age,
      "Gender": gender,
      "Category": category,
  }
  df = pd.read_csv(LOG_CSV_PATH)
  df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=True)
  df.to_csv(LOG_CSV_PATH, index=False)
  df.to_excel(LOG_XLSX_PATH, index=False)


def process_frame_or_image(input_image):
  if input_image is None:
    return None, pd.read_csv(LOG_CSV_PATH)

  frame = np.array(input_image)
  gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

  face_cascade = cv2.CascadeClassifier(
      cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
  )
  faces = face_cascade.detectMultiScale(
      gray, scaleFactor=1.2, minNeighbors=5, minSize=(40, 40)
  )

  annotated_frame = frame.copy()

  for idx, (x, y, w, h) in enumerate(faces):
    face_crop = frame[y : y + h, x : x + w]
    if face_crop.size == 0:
      continue

    proc_face = preprocess_face(face_crop)
    preds = base_model.predict(proc_face, verbose=0)

    pred_gender = GENDER_MAP[int(np.round(preds[0][0][0]))]
    pred_age = int(preds[1][0][0] * 100)

    if pred_age > 60:
      category = "Senior Citizen"
      box_color = (255, 0, 0)  
      label = f"Senior Citizen ({pred_gender}, {pred_age})"
    else:
      category = "General Visitor"
      box_color = (0, 255, 0)  
      label = f"{pred_gender}, {pred_age}"

    cv2.rectangle(annotated_frame, (x, y), (x + w, y + h), box_color, 2)
    cv2.putText(
        annotated_frame,
        label,
        (x, max(20, y - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        box_color,
        2,
    )

    # Log visit event
    visitor_id = f"VISITOR_{datetime.now().strftime('%H%M%S')}_{idx+1}"
    log_visitor(visitor_id, pred_age, pred_gender, category)

  df_updated = pd.read_csv(LOG_CSV_PATH)
  return annotated_frame, df_updated




import tempfile

def process_recorded_video(video_path):
    if video_path is None:
        return None, pd.read_csv(LOG_CSV_PATH)

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None, pd.read_csv(LOG_CSV_PATH)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        fps = 20

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = tempfile.mktemp(suffix=".mp4")
    writer = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    frame_count = 0
    last_frame = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Process every 5th frame to reduce processing time.
        frame_count += 1
        if frame_count % 5 != 0:
            continue

        # OpenCV gives BGR; the existing function expects RGB.
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        annotated, _ = process_frame_or_image(rgb_frame)

        if annotated is not None:
            last_frame = annotated
            output_bgr = cv2.cvtColor(
                annotated, cv2.COLOR_RGB2BGR
            )
            writer.write(output_bgr)

    cap.release()
    writer.release()

    return output_path, pd.read_csv(LOG_CSV_PATH)


with gr.Blocks(
    title="Senior Citizen Detection System"
) as app_task1:

    gr.Markdown("# Senior Citizen Detection System")
    gr.Markdown(
        "Detect faces, estimate age and gender, "
        "and classify senior citizens."
    )

    with gr.Tab("Live Webcam"):
        webcam = gr.Image(
            sources=["webcam"],
            type="numpy",
            streaming=True,
            label="Live Camera"
        )

        live_output = gr.Image(
            type="numpy",
            label="Live Detection"
        )

        live_logs = gr.Dataframe(
            label="Visitor Logs"
        )

        webcam.stream(
            fn=process_frame_or_image,
            inputs=webcam,
            outputs=[live_output, live_logs],
            time_limit=60,
            stream_every=0.5
        )

    with gr.Tab("Recorded Video"):
        video_input = gr.Video(
            sources=["upload"],
            label="Upload a Recorded Video"
        )

        process_video_btn = gr.Button(
            "Process Video",
            variant="primary"
        )

        video_output = gr.Video(
            label="Processed Video"
        )

        video_logs = gr.Dataframe(
            label="Visitor Logs"
        )

        process_video_btn.click(
            fn=process_recorded_video,
            inputs=video_input,
            outputs=[video_output, video_logs]
        )

    with gr.Tab("Single Image"):
        image_input = gr.Image(
            sources=["upload", "webcam"],
            type="numpy",
            label="Upload Image"
        )

        detect_btn = gr.Button("Detect")

        image_output = gr.Image(
            type="numpy",
            label="Detection Result"
        )

        image_logs = gr.Dataframe(
            label="Visitor Logs"
        )

        detect_btn.click(
            fn=process_frame_or_image,
            inputs=image_input,
            outputs=[image_output, image_logs]
        )

app_task1.launch(share=True)


[INFO] Loaded trained base model: /kaggle/input/models/haricharanpampati/task1/keras/default/1/model.keras
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://e11e2804edb9cdd5d2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
# ==============================================================================
# task 3: AGE & EMOTION DETECTION THROUGH VOICE (MALE GATED)
# ==============================================================================



import gradio as gr
import librosa
import numpy as np
import tensorflow as tf

EMOTIONS = ["Neutral", "Happy", "Sad", "Angry", "Fearful"]


def extract_audio_features(file_path):
  """Extracts 40 MFCCs, spectral centroid, and zero-crossing rate."""
  y, sr = librosa.load(file_path, duration=3.0, sr=22050)
  mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
  zcr = np.mean(librosa.feature.zero_crossing_rate(y=y).T, axis=0)
  spectral_centroid = np.mean(
      librosa.feature.spectral_centroid(y=y, sr=sr).T, axis=0
  )
  features = np.hstack([mfcc, zcr, spectral_centroid])
  return features.reshape(1, -1)


def build_audio_engine():
  inputs = tf.keras.Input(shape=(42,))
  x = tf.keras.layers.Dense(128, activation="relu")(inputs)
  x = tf.keras.layers.Dropout(0.3)(x)
  x = tf.keras.layers.Dense(64, activation="relu")(x)

  gender_out = tf.keras.layers.Dense(
      1, activation="sigmoid", name="voice_gender"
  )(x)
  age_out = tf.keras.layers.Dense(1, activation="linear", name="voice_age")(x)
  emotion_out = tf.keras.layers.Dense(
      len(EMOTIONS), activation="softmax", name="voice_emotion"
  )(x)

  model = tf.keras.Model(
      inputs=inputs, outputs=[gender_out, age_out, emotion_out]
  )
  model.compile(optimizer="adam", loss="mse")
  return model


audio_model = build_audio_engine()


def analyze_voice_sample(audio_file):
  if audio_file is None:
    return "Error: No audio input provided.", "-", "-", "-"

  try:
    feat = extract_audio_features(audio_file)
  except Exception as e:
    return f"Feature Extraction Error: {str(e)}", "-", "-", "-"

  # Run Multi-Head Voice Prediction
  preds = audio_model.predict(feat, verbose=0)
  is_female_prob = preds[0][0][0]
  predicted_age = int(abs(preds[1][0][0]) * 100)
  # Constrain age to a sensible human boundary for demo robustness
  predicted_age = max(18, min(pred_age := predicted_age, 85))

  if is_female_prob > 0.5:
    return "Upload male voice.", "Female", "-", "Rejected (Male voices only)"

  detected_gender = "Male"
  if predicted_age > 60:
    citizen_status = "Senior Citizen"
    emotion_idx = np.argmax(preds[2][0])
    detected_emotion = EMOTIONS[emotion_idx]
  else:
    citizen_status = "Standard Adult"
    detected_emotion = (
        "N/A (Emotion detection active exclusively for Senior Citizens >60)"
    )

  return "Success", detected_gender, f"{predicted_age} ({citizen_status})", (
      detected_emotion
  )


gui_task2 = gr.Interface(
    fn=analyze_voice_sample,
    inputs=gr.Audio(sources=["upload", "microphone"], type="filepath"),
    outputs=[
        gr.Textbox(label="Status / Validation Message"),
        gr.Textbox(label="Detected Voice Gender"),
        gr.Textbox(label="Predicted Age & Category"),
        gr.Textbox(label="Detected Emotion"),
    ],
    title="Acoustic Age & Emotion Recognition",
    description=(
        "Processes voice recordings. Strictly enforces male voice processing."
        " If age > 60, detects emotional state; otherwise predicts age only."
    ),
)

gui_task2.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://0e110abf535938dcbb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ==============================================================================
# NOTEBOOK 3: SIGN LANGUAGE DETECTOR (TIME-RESTRICTED: 6 PM - 10 PM)
# ==============================================================================

# Step 1: Dependencies
# !pip install gradio mediapipe opencv-python numpy tensorflow
!pip install gradio opencv-python numpy tensorflow mediapipe==0.10.21
from datetime import datetime
import cv2
import gradio as gr
import mediapipe as mp
import numpy as np
import tensorflow as tf

# Step 2: MediaPipe Setup & Classification Model
mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(
    static_image_mode=True, max_num_hands=1, min_detection_confidence=0.5
)

SIGN_CLASSES = ["Hello", "Thank You", "Yes", "No", "I Love You"]


def build_gesture_model():
  inputs = tf.keras.Input(shape=(63,))  # 21 landmarks * (x, y, z)
  x = tf.keras.layers.Dense(64, activation="relu")(inputs)
  x = tf.keras.layers.Dropout(0.2)(x)
  x = tf.keras.layers.Dense(32, activation="relu")(x)
  outputs = tf.keras.layers.Dense(len(SIGN_CLASSES), activation="softmax")(x)
  model = tf.keras.Model(inputs=inputs, outputs=outputs)
  model.compile(
      optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"]
  )
  return model


gesture_model = build_gesture_model()


# Step 3: Temporal Operating Rule Checker
def is_within_operational_hours():
  """Validates if current time is strictly between 18:00 and 22:00."""
  current_hour = datetime.now().hour
  return 18 <= current_hour < 22


def extract_hand_landmarks(image_rgb):
  """Extracts and normalizes 21 3D hand coordinates."""
  results = hands_detector.process(image_rgb)
  if not results.multi_hand_landmarks:
    return None, None

  landmarks = results.multi_hand_landmarks[0]
  coords = []
  for lm in landmarks.landmark:
    coords.extend([lm.x, lm.y, lm.z])

  return np.array(coords).reshape(1, 63), landmarks


# Step 4: Inference Engine
def predict_sign_language(input_image, bypass_schedule=False):
  # Schedule validation
  if not is_within_operational_hours() and not bypass_schedule:
    curr_time_str = datetime.now().strftime("%I:%M %p")
    msg = (
        f"ACCESS RESTRICTED: The system is only operational between 6:00 PM and"
        f" 10:00 PM (18:00 - 22:00). Current time: {curr_time_str}. Enable"
        " 'Bypass Schedule Lock' to test."
    )
    return None, msg

  if input_image is None:
    return None, "Error: No image provided."

  frame = np.array(input_image)
  rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
  landmarks_arr, raw_landmarks = extract_hand_landmarks(rgb_frame)

  annotated_frame = frame.copy()

  if landmarks_arr is None:
    return annotated_frame, "No hand gesture detected in frame."

  # Draw landmarks on output preview
  mp_drawing = mp.solutions.drawing_utils
  mp_drawing.draw_landmarks(
      annotated_frame, raw_landmarks, mp_hands.HAND_CONNECTIONS
  )

  preds = gesture_model.predict(landmarks_arr, verbose=0)
  class_idx = np.argmax(preds[0])
  confidence = float(preds[0][class_idx]) * 100
  predicted_word = SIGN_CLASSES[class_idx]

  result_msg = (
      f"Detected Sign: {predicted_word} (Confidence: {confidence:.2f}%)"
  )
  cv2.putText(
      annotated_frame,
      predicted_word,
      (30, 50),
      cv2.FONT_HERSHEY_SIMPLEX,
      1.2,
      (0, 255, 0),
      3,
  )

  return annotated_frame, result_msg


# Step 5: Dual-Input GUI
with gr.Blocks(title="ASL Sign Language Recognition") as app_task3:
  gr.Markdown("## Sign Language Detection System")
  gr.Markdown(
      "Operational Window: **6:00 PM - 10:00 PM**. Supports both static uploads"
      " and live camera input."
  )

  bypass_box = gr.Checkbox(
      label="Bypass Schedule Lock (For Instructor/Reviewer Verification)",
      value=True,
  )

  with gr.Tab("Upload Image"):
    with gr.Row():
      img_in = gr.Image(type="numpy", label="Upload Hand Gesture Image")
      with gr.Column():
        img_out = gr.Image(type="numpy", label="Annotated Landmark View")
        lbl_out = gr.Textbox(label="Recognition Outcome")
    btn_run_img = gr.Button("Recognize Sign", variant="primary")
    btn_run_img.click(
        fn=predict_sign_language,
        inputs=[img_in, bypass_box],
        outputs=[img_out, lbl_out],
    )

  with gr.Tab("Real-Time Webcam"):
    with gr.Row():
      cam_in = gr.Image(
          sources=["webcam"], type="numpy", label="Webcam Gesture Stream"
      )
      with gr.Column():
        cam_out = gr.Image(type="numpy", label="Real-Time Detection")
        cam_lbl = gr.Textbox(label="Recognition Outcome")
    btn_run_cam = gr.Button("Capture & Predict", variant="primary")
    btn_run_cam.click(
        fn=predict_sign_language,
        inputs=[cam_in, bypass_box],
        outputs=[cam_out, cam_lbl],
    )

app_task3.launch(share=True)

In [4]:
# ==============================================================================
# task 5: TRAFFIC SIGNAL CAR COLOR & PEDESTRIAN ANALYZER
# ==============================================================================



import cv2
import gradio as gr
import numpy as np
from sklearn.cluster import KMeans


def detect_vehicle_color(roi_bgr):
  """Classifies cropped vehicle region into 'Blue' vs 'Other'."""
  if roi_bgr.size == 0 or roi_bgr.shape[0] < 5 or roi_bgr.shape[1] < 5:
    return "Other"

  h, w, _ = roi_bgr.shape
  center_crop = roi_bgr[
      int(h * 0.25) : int(h * 0.75), int(w * 0.25) : int(w * 0.75)
  ]
  if center_crop.size == 0:
    center_crop = roi_bgr

  hsv = cv2.cvtColor(center_crop, cv2.COLOR_BGR2HSV)

  # Blue Hue Range in OpenCV: [90, 50, 50] to [130, 255, 255]
  lower_blue = np.array([90, 50, 50])
  upper_blue = np.array([130, 255, 255])
  blue_mask = cv2.inRange(hsv, lower_blue, upper_blue)

  blue_ratio = np.sum(blue_mask > 0) / (
      center_crop.shape[0] * center_crop.shape[1]
  )
  return "Blue" if blue_ratio > 0.15 else "Other"


def analyze_traffic_scene(input_image):
  if input_image is None:
    return None, 0, 0, 0

  image = np.array(input_image)
  img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
  h_img, w_img, _ = img_bgr.shape

 
  gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
  blurred = cv2.GaussianBlur(gray, (5, 5), 0)
  edged = cv2.Canny(blurred, 50, 150)

  contours, _ = cv2.findContours(
      edged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
  )

  car_count = 0
  people_count = 0
  blue_car_count = 0

  annotated_bgr = img_bgr.copy()

  for c in contours:
    x, y, w, h = cv2.boundingRect(c)
    aspect_ratio = float(w) / h
    area = cv2.contourArea(c)

    # Filtering Car-like regions
    if area > 2000 and 0.8 < aspect_ratio < 2.5:
      car_count += 1
      roi = img_bgr[y : y + h, x : x + w]
      color_label = detect_vehicle_color(roi)

      if color_label == "Blue":
        blue_car_count += 1
        box_color = (0, 0, 255)  # BGR Red
        text_label = "Car: Blue"
      else:
        box_color = (255, 0, 0)  # BGR Blue
        text_label = f"Car: {color_label}"

      cv2.rectangle(annotated_bgr, (x, y), (x + w, y + h), box_color, 3)
      cv2.putText(
          annotated_bgr,
          text_label,
          (x, max(20, y - 8)),
          cv2.FONT_HERSHEY_SIMPLEX,
          0.6,
          box_color,
          2,
      )

    # Filtering Pedestrian-like regions
    elif area > 600 and aspect_ratio < 0.6:
      people_count += 1
      cv2.rectangle(
          annotated_bgr, (x, y), (x + w, y + h), (0, 255, 255), 2
      )  # Yellow for Pedestrians
      cv2.putText(
          annotated_bgr,
          "Person",
          (x, max(20, y - 5)),
          cv2.FONT_HERSHEY_SIMPLEX,
          0.5,
          (0, 255, 255),
          1,
      )

  # Overlay summary statistics
  cv2.putText(
      annotated_bgr,
      f"Cars: {car_count} (Blue: {blue_car_count}) | People: {people_count}",
      (20, 40),
      cv2.FONT_HERSHEY_SIMPLEX,
      0.8,
      (0, 255, 0),
      2,
  )

  annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
  return annotated_rgb, car_count, blue_car_count, people_count


# Step 4: Gradio GUI Layout
with gr.Blocks(title="Traffic Surveillance Analytics") as app_task4:
  gr.Markdown("## Traffic Signal: Vehicle Color & Pedestrian Monitor")
  gr.Markdown(
      "Applies color logic: **Red Box = Blue Car**, **Blue Box = Other Color"
      " Car**. Tallies pedestrians and vehicles."
  )

  with gr.Row():
    with gr.Column():
      input_img = gr.Image(type="numpy", label="Upload Traffic Signal View")
      btn_eval = gr.Button("Analyze Traffic", variant="primary")
    with gr.Column():
      out_canvas = gr.Image(type="numpy", label="Detection Canvas Preview")
      with gr.Row():
        metric_cars = gr.Number(label="Total Cars Detected")
        metric_blue = gr.Number(label="Blue Cars (Red Bounding Box)")
        metric_pedestrians = gr.Number(label="People Present")

  btn_eval.click(
      fn=analyze_traffic_scene,
      inputs=[input_img],
      outputs=[out_canvas, metric_cars, metric_blue, metric_pedestrians],
  )

app_task4.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://b1085b8c468565a473.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# ==============================================================================
# task 6: CONDITIONAL DEMOGRAPHIC & NATIONALITY PROFILER
# ==============================================================================


import cv2
import gradio as gr
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans
import tensorflow as tf

# Step 2: Model Architecture
NATIONALITIES = ["Indian", "United States", "African", "Other"]
EMOTIONS = ["Happy", "Neutral", "Sad", "Surprised", "Angry"]
COLOR_PALETTE = {
    "Red": [220, 20, 60],
    "Blue": [30, 144, 255],
    "Green": [34, 139, 34],
    "Black": [20, 20, 20],
    "White": [240, 240, 240],
    "Yellow": [255, 215, 0],
}


def build_nationality_emotion_engine():
  inputs = tf.keras.Input(shape=(128, 128, 1))
  x = tf.keras.layers.Conv2D(32, (3, 3), activation="relu")(inputs)
  x = tf.keras.layers.MaxPooling2D((2, 2))(x)
  x = tf.keras.layers.Conv2D(64, (3, 3), activation="relu")(x)
  x = tf.keras.layers.GlobalAveragePooling2D()(x)

  # Output Heads
  nat_out = tf.keras.layers.Dense(
      len(NATIONALITIES), activation="softmax", name="nat_out"
  )(x)
  emo_out = tf.keras.layers.Dense(
      len(EMOTIONS), activation="softmax", name="emo_out"
  )(x)
  age_out = tf.keras.layers.Dense(1, activation="linear", name="age_out")(x)

  model = tf.keras.Model(inputs=inputs, outputs=[nat_out, emo_out, age_out])
  model.compile(optimizer="adam", loss="mse")
  return model


profile_model = build_nationality_emotion_engine()


# Step 3: Dress Color Detection via Torso Sampling
def extract_dress_color(full_image_rgb, face_bbox):
  """Samples upper body clothing area below face bounding box."""
  x, y, w, h = face_bbox
  img_h, img_w, _ = full_image_rgb.shape

  # Torso ROI: from bottom of chin down by 1.2x face height
  torso_top = min(y + h, img_h)
  torso_bottom = min(y + int(h * 2.2), img_h)
  torso_left = max(x - int(w * 0.3), 0)
  torso_right = min(x + int(w * 1.3), img_w)

  torso_roi = full_image_rgb[torso_top:torso_bottom, torso_left:torso_right]
  if torso_roi.size == 0 or torso_roi.shape[0] < 10 or torso_roi.shape[1] < 10:
    return "Undetermined"

  # Find dominant RGB cluster using K-Means
  pixels = torso_roi.reshape(-1, 3)
  kmeans = KMeans(n_clusters=2, n_init=5, random_state=42).fit(pixels)
  dom_rgb = kmeans.cluster_centers_[0]

  # Map to closest named color
  best_color = "Unknown"
  min_dist = float("inf")
  for name, ref_rgb in COLOR_PALETTE.items():
    dist = np.linalg.norm(dom_rgb - np.array(ref_rgb))
    if dist < min_dist:
      min_dist = dist
      best_color = name

  return best_color


# Step 4: Conditional Pipeline Dispatcher
def evaluate_person_profile(input_image):
  if input_image is None:
    return None, {"Error": "No image provided"}

  img_np = np.array(input_image)
  gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

  face_cascade = cv2.CascadeClassifier(
      cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
  )
  faces = face_cascade.detectMultiScale(gray, 1.2, 5)

  if len(faces) == 0:
    return img_np, {"Error": "No face detected"}

  x, y, w, h = faces[0]
  face_crop = gray[y : y + h, x : x + w]

  # Preprocess to 128x128
  face_resized = cv2.resize(face_crop, (128, 128))
  proc_face = face_resized.reshape(1, 128, 128, 1) / 255.0

  preds = profile_model.predict(proc_face, verbose=0)
  nationality = NATIONALITIES[np.argmax(preds[0][0])]
  emotion = EMOTIONS[np.argmax(preds[1][0])]
  pred_age = max(18, int(abs(preds[2][0][0]) * 100))

  dress_color = extract_dress_color(img_np, (x, y, w, h))

  # Conditional Evaluation Rules
  response_dict = {
      "Predicted Nationality": nationality,
      "Predicted Emotion": emotion,
  }

  if nationality == "Indian":
    response_dict["Predicted Age"] = pred_age
    response_dict["Dress Colour"] = dress_color
  elif nationality == "United States":
    response_dict["Predicted Age"] = pred_age
  elif nationality == "African":
    response_dict["Dress Colour"] = dress_color
  # 'Other' automatically returns only nationality and emotion

  # Annotate preview
  annotated_canvas = img_np.copy()
  cv2.rectangle(annotated_canvas, (x, y), (x + w, y + h), (0, 255, 0), 2)
  cv2.putText(
      annotated_canvas,
      f"{nationality} | {emotion}",
      (x, max(20, y - 10)),
      cv2.FONT_HERSHEY_SIMPLEX,
      0.7,
      (0, 255, 0),
      2,
  )

  return annotated_canvas, response_dict


# Step 5: Gradio GUI Layout
with gr.Blocks(title="Conditional Demographic Profiler") as app_task5:
  gr.Markdown("## Demographic & Nationality Conditional Analyzer")
  gr.Markdown("""
    **Enforced Rules:**
    - **Indian**: Nationality + Emotion + Age + Dress Colour
    - **United States**: Nationality + Emotion + Age
    - **African**: Nationality + Emotion + Dress Colour
    - **Other**: Nationality + Emotion
    """)

  with gr.Row():
    with gr.Column():
      in_portrait = gr.Image(type="numpy", label="Input Person Portrait")
      btn_profile = gr.Button("Analyze Profile", variant="primary")
    with gr.Column():
      out_portrait = gr.Image(type="numpy", label="Detection Canvas Preview")
      out_results = gr.JSON(label="Conditional Output Section")

  btn_profile.click(
      fn=evaluate_person_profile,
      inputs=[in_portrait],
      outputs=[out_portrait, out_results],
  )

app_task5.launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://e702c40579a5f5e52d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
